# Qué es un embedding

Antes de montar un RAG conviene ver la pieza sobre la que se apoya todo: convertir texto
en un vector de números que captura su **significado**, de forma que textos parecidos
queden cerca en el espacio vectorial.

## Objetivos

1. Convertir varios textos en vectores con `embed_documents`.
2. Ver la **dimensión** de un embedding y qué aspecto tiene.
3. Entender la diferencia entre `embed_documents` y `embed_query`.

## Qué hacemos

```mermaid
flowchart TB
    subgraph DOCS["Indexar · lo que guardas"]
        A["5 frases de una conversación"] --> B["☁️ OpenAI · embed_documents<br/>text-embedding-3-small"]
        B --> C["5 vectores<br/>1536 dimensiones cada uno"]
    end

    subgraph QRY["Consultar · lo que preguntas"]
        D["¿cuál es el nombre mencionado?"] --> E["☁️ OpenAI · embed_query<br/>text-embedding-3-small"]
        E --> F["1 vector<br/>1536 dimensiones"]
    end

    C -.->|distancia coseno| F

    classDef openai fill:#10a37f,stroke:#0b6e55,color:#ffffff,stroke-width:2px
    class B,E openai
```

**La clave:** ambos métodos producen vectores del mismo tamaño y en el mismo espacio,
que es lo que permite compararlos. Existen por separado porque algunos modelos tratan
distinto un documento que se archiva y una pregunta que se busca; en
`text-embedding-3-small` el resultado es equivalente, pero conviene usar cada uno en su sitio.


## 1 · Setup

Imports y API keys desde el `.env` de la raíz del repo.


In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import requests

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings


/var/folders/ym/1598v2z14n38j4znw7slpmpc0000gn/T/ipykernel_21624/1502158227.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
EMBEDDINGS = OpenAIEmbeddings(model="text-embedding-3-small")

## 2 · Vectorizar varios textos

`embed_documents` recibe una **lista** de textos y devuelve una lista de vectores,
en el mismo orden. Es lo que usa por dentro un vector store al indexar.

Las cinco frases son una conversación mínima: dos saludos, una pregunta por el nombre,
la respuesta y un saludo personalizado.


In [3]:
embeddings_list= EMBEDDINGS.embed_documents(
    [
        'Hola!',
        'Hola, ¿cómo estas?',
        '¿cuál es tu nombre?',
        'Me llamo Pepito',
        'Hola Pepito'
    ]
)

In [4]:
embeddings_list

[[0.036529541015625,
  -0.0283050537109375,
  0.0189971923828125,
  0.0182037353515625,
  0.0094757080078125,
  -0.01514434814453125,
  -0.025360107421875,
  0.035980224609375,
  -0.025360107421875,
  -0.056671142578125,
  -0.00537109375,
  -0.0209197998046875,
  0.0178985595703125,
  -0.03448486328125,
  -0.0027675628662109375,
  0.034759521484375,
  -0.048675537109375,
  -0.018096923828125,
  -0.030975341796875,
  0.0721435546875,
  0.04742431640625,
  -0.019256591796875,
  -0.021087646484375,
  0.005130767822265625,
  0.0244140625,
  0.00702667236328125,
  0.0105133056640625,
  -0.0020236968994140625,
  0.037933349609375,
  -0.0056610107421875,
  0.06396484375,
  -0.03302001953125,
  -0.005512237548828125,
  -0.008453369140625,
  -0.018798828125,
  0.0147552490234375,
  -0.01904296875,
  0.03350830078125,
  0.00634002685546875,
  -0.018524169921875,
  0.024871826171875,
  -0.0185089111328125,
  -0.01947021484375,
  0.0206298828125,
  -0.004550933837890625,
  -0.0035953521728515625,


### ¿Qué tamaño tiene un embedding?

`text-embedding-3-small` produce vectores de **1536 dimensiones**. Ese número es fijo:
da igual que el texto sea `"Hola!"` o un párrafo entero, el vector siempre mide lo mismo.
Eso es justo lo que permite compararlos entre sí.


In [5]:
len(embeddings_list[0])

1536

## 3 · Vectorizar una consulta

`embed_query` es el equivalente para el lado de la búsqueda: recibe **un solo** texto
y devuelve **un solo** vector.

Fíjate en que la pregunta *"¿cuál es el nombre mencionado en la conversación?"* no comparte
casi ninguna palabra con *"Me llamo Pepito"*. Por eso la búsqueda por palabras clave
fallaría aquí y la búsqueda por embeddings no: compara significado, no letras.


In [6]:
EMBEDDINGS.embed_query('¿cuál es el nombre mencionado en la conversación')

[0.0163421630859375,
 0.01055908203125,
 -0.0120391845703125,
 0.0018415451049804688,
 -0.00453948974609375,
 0.038238525390625,
 -0.0308990478515625,
 0.048095703125,
 0.001617431640625,
 -0.0931396484375,
 0.035552978515625,
 0.0008134841918945312,
 0.0014276504516601562,
 -0.046539306640625,
 -0.01366424560546875,
 0.00478363037109375,
 -0.05010986328125,
 -0.0207672119140625,
 -0.003673553466796875,
 0.00661468505859375,
 -0.023406982421875,
 0.0640869140625,
 0.00485992431640625,
 0.045867919921875,
 -0.035430908203125,
 -0.0210418701171875,
 -0.03912353515625,
 0.02459716796875,
 0.0004277229309082031,
 -0.030364990234375,
 0.009521484375,
 -0.044281005859375,
 -0.006786346435546875,
 0.0166015625,
 -0.0360107421875,
 6.395578384399414e-05,
 0.01451873779296875,
 -0.038116455078125,
 0.0031261444091796875,
 -0.01265716552734375,
 -0.018096923828125,
 -0.0245513916015625,
 -0.0270538330078125,
 0.0219573974609375,
 0.0307159423828125,
 0.0113372802734375,
 -0.0273590087890625,
 -0